In [12]:
from bs4 import BeautifulSoup

HTML_CONTOH = """
<!doctype html>
<html lang="id">
  <head><title>Toko Buku Data</title></head>
  <body>
    <h1>Buku Pilihan</h1>
    <section id="katalog">
      <article class="buku unggulan" data-id="B001">
        <h2 class="judul">Dasar Data Mining</h2>
        <p class="harga">Rp125.000</p>
        <p class="stok">Tersedia</p>
        <a href="/buku/dasar-data-mining">Detail</a>
      </article>
      <article class="buku" data-id="B002">
        <h2 class="judul">Python untuk Analisis Data</h2>
        <p class="harga">Rp149.500</p>
        <p class="stok habis">Habis</p>
        <a href="/buku/python-analisis">Detail</a>
      </article>
      <article class="buku" data-id="B003">
        <h2 class="judul">Statistika Praktis</h2>
        <p class="harga">Rp98.000</p>
        <!-- Elemen stok sengaja tidak tersedia -->
        <a href="/buku/statistika-praktis">Detail</a>
      </article>
    </section>
  </body>
</html>
"""

soup = BeautifulSoup(HTML_CONTOH, "html.parser")

# ubah harga jadi angka
def harga_ke_int(teks_harga):
    if teks_harga is None:
        return None

    digit = ""
    for karakter in teks_harga:
        if karakter.isdigit():
            digit += karakter

    return int(digit) if digit else None

# 1. ambil judul buku yang statusnya Tersedia
judul_tersedia = []

for buku in soup.select("article.buku"):
    stok = buku.select_one(".stok")

    if stok and stok.get_text(strip=True) == "Tersedia":
        judul = buku.select_one(".judul").get_text(strip=True)
        judul_tersedia.append(judul)

print("Judul buku yang tersedia:", judul_tersedia)

# 2. hitung rata-rata harga buku
harga = []

for buku in soup.select("article.buku"):
    teks_harga = buku.select_one(".harga").get_text(strip=True)
    harga.append(harga_ke_int(teks_harga))

rata_rata = sum(harga)/len(harga)

print("Rata rata harga buku:", rata_rata)

# 3. ambil semua atribut data-id dengan satu CSS selector
data_id = []

for buku in soup.select("article.buku"):
    data_id.append(buku.get("data-id"))

print("Semua data-id:", data_id)

# 4. ubah fungsi ekstraksi agar stok yang hilang menjadi "Tidak diketahui"
def teks_atau_none(induk, selector):
    elemen = induk.select_one(selector)

    if elemen:
        return elemen.get_text(" ", strip=True)

    return None

def ekstrak_buku(dokumen):
    hasil = []

    for buku in dokumen.select("article.buku"):
        hasil.append({
            "id": buku.get("data-id"),
            "judul": teks_atau_none(buku, ".judul"),
            "harga_rupiah": harga_ke_int(teks_atau_none(buku, ".harga")),
            "stok": teks_atau_none(buku, ".stok") or "Tidak diketahui"
        })

    return hasil

print(ekstrak_buku(soup))

Judul buku yang tersedia: ['Dasar Data Mining']
Rata rata harga buku: 124166.66666666667
Semua data-id: ['B001', 'B002', 'B003']
[{'id': 'B001', 'judul': 'Dasar Data Mining', 'harga_rupiah': 125000, 'stok': 'Tersedia'}, {'id': 'B002', 'judul': 'Python untuk Analisis Data', 'harga_rupiah': 149500, 'stok': 'Habis'}, {'id': 'B003', 'judul': 'Statistika Praktis', 'harga_rupiah': 98000, 'stok': 'Tidak diketahui'}]
